# 第二周作业 数据预处理
任务背景：本项目提供了来自某在线学习平台的学生学习数据，包括studentInfo.csv（学生基本信息）、studentAssessment.csv（学生评分记录）和assessment.csv（评分标准）三张表。部分数据项解释如下：
- code_module：课程编号
- code_presentation：开课学期
- id_student：学生id
- id_assessment：评分id

任务描述：
1. 读取studentInfo.csv和studentAssessment.csv。检查这两张表的数据规模和缺失值情况。
2. 在studentInfo表中，找到imd_band（贫困指数）的缺失值，选择合适的值进行填补；在studentAssessment表中，有些学生可能缺考导致score列为空，请将这些缺失的成绩填充为0。
3. 根据assessments表中信息，筛选studentAssessment表中所有类型为考试（Exam）的评分记录，并计算出每个学生的平均成绩（mean_score），将结果保存为一个新的数据框。
4. 将计算好的平均成绩表与studentInfo表进行连接，生成一张既包含学生人口特征，又包含其平均考试成绩的表。

最终提交：数据处理程序代码（本文件）。

本任务允许且鼓励你使用AI工具辅助完成，如果使用，在下方AI对话记录板块粘贴对话链接。

In [1]:
import pandas as pd

# 读取数据
student_info = pd.read_csv("studentInfo.csv", encoding="gb18030")
student_assessment = pd.read_csv("studentAssessment.csv")
assessments = pd.read_csv("assessments.csv")

# 删除 studentAssessment 中无效的空列名列
student_assessment = student_assessment.loc[:, ~student_assessment.columns.str.contains(r"^Unnamed")]
student_assessment = student_assessment.loc[:, student_assessment.columns != ""]

# 第一题：查看数据规模和缺失值情况
print("studentInfo shape:", student_info.shape)
print("studentAssessment shape:", student_assessment.shape)
print("assessments shape:", assessments.shape)
print("\nstudentInfo columns:")
print(student_info.columns.tolist())
print("\nstudentAssessment columns:")
print(student_assessment.columns.tolist())
print("\nassessments columns:")
print(assessments.columns.tolist())
print("\nstudentInfo missing values:")
print(student_info.isna().sum())
print("\nstudentAssessment missing values:")
print(student_assessment.isna().sum())

# 第二题：填补缺失值
# studentInfo 中的 imd_band 是分类变量，用众数填补更合适
student_info["imd_band"] = student_info["imd_band"].replace("10月20日", "10-20%")
imd_mode = student_info["imd_band"].mode()[0]
student_info["imd_band"] = student_info["imd_band"].fillna(imd_mode)

# studentAssessment 中缺考学生的 score 按题意填充为 0
student_assessment["score"] = student_assessment["score"].fillna(0)

print("\nfilled imd_band with:", imd_mode)
print("studentInfo imd_band missing after fill:", student_info["imd_band"].isna().sum())
print("studentAssessment score missing after fill:", student_assessment["score"].isna().sum())


studentInfo shape: (32593, 12)
studentAssessment shape: (173912, 5)
assessments shape: (206, 6)

studentInfo columns:
['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result']

studentAssessment columns:
['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']

assessments columns:
['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']

studentInfo missing values:
code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64

studentAssessment missing values:
id_assessment       0
id_student          0
date_submit

In [2]:
# 第三题：筛选 Exam 类型的评分记录，并计算每个学生的平均考试成绩
exam_assessments = assessments[assessments["assessment_type"] == "Exam"]

exam_scores = student_assessment.merge(
    exam_assessments[["id_assessment", "assessment_type"]],
    on="id_assessment",
    how="inner"
)

mean_score_df = (
    exam_scores.groupby("id_student", as_index=False)["score"]
    .mean()
    .rename(columns={"score": "mean_score"})
)

print("Exam records shape:", exam_scores.shape)
print(mean_score_df.head())


Exam records shape: (4959, 6)
   id_student  mean_score
0       23698        80.0
1       24213        58.0
2       27116        96.0
3       28046        40.0
4       28787        44.0


## AI对话记录
- AI对话链接：

In [3]:
# 第四题：将平均考试成绩表与 studentInfo 表连接，并导出结果
final_df = student_info.merge(mean_score_df, on="id_student", how="left")

print("final_df shape:", final_df.shape)
print(final_df.head())

final_df.to_csv("student_info_with_mean_score.csv", index=False, encoding="utf-8-sig")
print("\n已导出文件：student_info_with_mean_score.csv")


final_df shape: (32593, 13)
  code_module code_presentation  id_student gender                region  \
0         AAA             2013J       11391      M   East Anglian Region   
1         AAA             2013J       28400      F              Scotland   
2         AAA             2013J       30268      F  North Western Region   
3         AAA             2013J       31604      F     South East Region   
4         AAA             2013J       32885      F  West Midlands Region   

       highest_education imd_band age_band  num_of_prev_attempts  \
0       HE Qualification  90-100%     55<=                     0   
1       HE Qualification   20-30%    35-55                     0   
2  A Level or Equivalent   30-40%    35-55                     0   
3  A Level or Equivalent   50-60%    35-55                     0   
4     Lower Than A Level   50-60%     0-35                     0   

   studied_credits disability final_result  mean_score  
0              240          N         Pass       